# POLITSKY24 Dataset Accessing and Testing

In [1]:
import os
import pandas as pd

from pathlib import Path

# CHANGE THIS TO THE PATH WHERE THE DATASET IS LOCATED
data_path = Path("D:\Thesis-related\POLITISKY24")

<>:7: SyntaxWarning: invalid escape sequence '\T'
<>:7: SyntaxWarning: invalid escape sequence '\T'
C:\Users\andre\AppData\Local\Temp\ipykernel_80864\754992423.py:7: SyntaxWarning: invalid escape sequence '\T'
  data_path = Path("D:\Thesis-related\POLITISKY24")


In [2]:
# Access the parquet files in the data path

parquet_files = list(data_path.glob("*.parquet"))
parquet_files

[WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_stance_relevancy_dataset (post-target entity pairs).parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_stance_relevancy_dataset (query-post stance relevancy pairs).parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/llm_annotated_full_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/llm_annotated_validation_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_network_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_post_history_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_post_list_for_stance_detection.parquet')]

In [3]:
import polars as pl

# Load each dataset manually into individual variables
user_network_df = pl.read_parquet(str(data_path / "user_network_dataset.parquet"))
user_post_history_df = pl.read_parquet(str(data_path / "user_post_history_dataset.parquet"))
user_post_list_df = pl.read_parquet(str(data_path / "user_post_list_for_stance_detection.parquet"))

# Stance label datasets
llm_full_stance_df = pl.read_parquet(str(data_path / "llm_annotated_full_user_stance_dataset.parquet"))
llm_val_stance_df = pl.read_parquet(str(data_path / "llm_annotated_validation_user_stance_dataset.parquet"))
human_val_stance_df = pl.read_parquet(str(data_path / "human_annotated_validation_user_stance_dataset.parquet"))

# Stance relevancy datasets (Note: surround filename strings in quotes)
human_val_relevancy_post_df = pl.read_parquet(
    str(data_path / "human_annotated_validation_stance_relevancy_dataset (post-target entity pairs).parquet")
)
human_val_relevancy_query_df = pl.read_parquet(
    str(data_path / "human_annotated_validation_stance_relevancy_dataset (query-post stance relevancy pairs).parquet")
)

## Dataset File Inspection

In [4]:
user_network_df.schema

Schema([('SourceUserId', Int64),
        ('TargetUserId', Int64),
        ('InteractionType', String),
        ('Count', Int64)])

In [5]:
user_network_df

SourceUserId,TargetUserId,InteractionType,Count
i64,i64,str,i64
9,9,"""Like""",1
9,9,"""Quote""",42
9,9,"""Repost""",4
9,10,"""Quote""",2
9,2549,"""Quote""",1
…,…,…,…
401391,398769,"""Like""",2
401391,398831,"""Quote""",1
401391,401391,"""Like""",3


In [6]:
# Check number of rows in user_network_df

user_network_df.shape[0]

1676895

In [7]:
user_post_history_df.schema

Schema([('PostId', Int64),
        ('UserId', Int64),
        ('PostTime', String),
        ('Content', String),
        ('Hashtags', List(String)),
        ('Languages', List(String)),
        ('Mentions', List(Int64)),
        ('IsRepost', Boolean),
        ('SourceUserId', Int64),
        ('IsQuote', Boolean),
        ('QuoteId', Int64),
        ('QuoteUserId', Int64),
        ('QuoteContent', String),
        ('IsReply', Boolean),
        ('ParentId', Int64),
        ('ParentUserId', Int64)])

In [13]:
user_post_history_df.shape[0]

18416787

In [8]:
user_post_history_df.select(pl.all().n_unique()).to_dicts()[0]

{'PostId': 14830231,
 'UserId': 8561,
 'PostTime': 18230517,
 'Content': 13138778,
 'Hashtags': 285770,
 'Languages': 1193,
 'Mentions': 88178,
 'IsRepost': 2,
 'SourceUserId': 217858,
 'IsQuote': 2,
 'QuoteId': 975337,
 'QuoteUserId': 105488,
 'QuoteContent': 910916,
 'IsReply': 2,
 'ParentId': 5472655,
 'ParentUserId': 291653}

In [12]:
# Check user post content 
user_post_history_df.select("Content").head()

Content
str
"""it’s a tragedy of the commons.…"
"""Notícia da @anonymous ""Xi Jin…"
"""Have you run it by the Godless…"
"""This! This is the energy we ne…"
"""WATCH: Trending Stories: Gas P…"


In [12]:
# Exact duplicate rows (list columns may need to be dropped/converted first)
user_post_history_df.is_duplicated().sum()

# Rows sharing a PostId
dupes = user_post_history_df.filter(pl.col("PostId").is_duplicated())
dupes.sort("PostId").head(20)

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64
18,16171,"""2023-12-03T01:11:35.122Z""","""The end of Moorevember coincid…",[],"[""en""]",[],true,7,false,-1,-1,"""""",false,-1,-1
18,71390,"""2023-12-04T22:26:06.295Z""","""The end of Moorevember coincid…",[],"[""en""]",[],true,7,false,-1,-1,"""""",false,-1,-1
47,9,"""2023-06-22T00:48:37.165Z""","""I used a Chrome/Edge extension…",[],[],[],false,9,false,-1,-1,"""""",false,15691770,343645
47,285903,"""2023-06-22T00:49:34.781Z""","""I used a Chrome/Edge extension…",[],[],[],true,9,false,-1,-1,"""""",false,15691770,343645
279,9,"""2023-07-06T17:04:05.906Z""","""Should be required that PD off…",[],"[""en""]",[],false,9,false,-1,-1,"""""",false,4244240,90714
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
999,141415,"""2023-10-03T01:52:13.292Z""","""Let's talk about how domestic …",[],"[""en""]",[],true,9,false,-1,-1,"""""",false,-1,-1
1000,90924,"""2023-10-03T01:47:26.282Z""","""Going along with Sheryl's post…",[],"[""en""]",[],true,9,false,-1,-1,"""""",true,999,9
1000,9,"""2023-10-03T00:58:13.600Z""","""Going along with Sheryl's post…",[],"[""en""]",[],false,9,false,-1,-1,"""""",true,999,9


In [13]:
user_post_history_df["IsReply"].value_counts()

IsReply,count
bool,u32
false,11509179
true,6907608


In [14]:
# Non-replies that have content in ParentUserId
falseReplies = user_post_history_df.filter(
    (pl.col("IsReply") == "false") & (pl.col("ParentUserId") != -1)
    )
falseReplies.head()

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64
10727556,235242,"""2024-11-19T23:11:37.530Z""","""Because a LOT of these people …",[],"[""en""]",[],false,235242,false,-1,-1,"""""",false,12143282,268404
7583443,163361,"""2024-11-19T23:10:37.854Z""","""I'm trying to get used to it""",[],"[""en""]",[],false,163361,false,-1,-1,"""""",false,8475193,183152
8501377,183843,"""2024-11-19T23:08:13.424Z""","""What do you expect from a lose…",[],"[""en""]",[],false,183843,false,-1,-1,"""""",false,3826921,81362
10102388,221773,"""2024-11-19T23:07:03.508Z""","""ty comrade ❤️ op is on some ve…",[],"[""en""]",[],false,221773,false,-1,-1,"""""",false,13234926,290109
8060256,171717,"""2024-11-19T23:05:44.594Z""","""""",[],"[""en""]",[],false,171717,false,-1,-1,"""""",false,17883018,395662


In [15]:
# Non-reposts all have SourceUserIds equal to UserId (original author)
falseRepost = user_post_history_df.filter(
    (pl.col("IsRepost") == "false") & (pl.col("SourceUserId") != -1)
    )
falseRepost.head()

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64
7840946,167406,"""2024-11-19T23:12:12.092Z""","""it’s a tragedy of the commons.…",[],"[""en""]",[],false,167406,false,-1,-1,"""""",true,15792425,345622
17079336,377198,"""2024-11-19T23:12:11Z""","""Notícia da @anonymous ""Xi Jin…",[],[],[],false,377198,false,-1,-1,"""""",false,-1,-1
9179962,196837,"""2024-11-19T23:12:11.298Z""","""Have you run it by the Godless…",[],"[""en""]",[],false,196837,false,-1,-1,"""""",true,15458353,336741
2108430,45089,"""2024-11-19T23:12:11.084Z""","""This! This is the energy we ne…",[],"[""en""]",[],false,45089,true,2109954,45144,"""WATCH: “You’re a baby shit.” …",false,-1,-1
4231604,90172,"""2024-11-19T23:12:10Z""","""WATCH: Trending Stories: Gas P…",[],[],[],false,90172,false,-1,-1,"""""",false,-1,-1


In [16]:
falseQuotes = user_post_history_df.filter(
    (pl.col("IsQuote") == "false") & (pl.col("QuoteUserId") != -1)
    )
falseQuotes.head()

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64


In [17]:
user_post_history_df.shape

(18416787, 16)

In [18]:
user_post_list_df.schema

Schema([('UserId', Int64), ('PostIdList', List(Int64))])

In [19]:
llm_full_stance_df.schema

Schema([('UserId', Int64),
        ('TargetEntity', String),
        ('Context', List(String)),
        ('LLMAnswerContent', String),
        ('LLMAnswerUsage', String),
        ('SourcePosts', List(String)),
        ('Spans', List(String)),
        ('Reason', String),
        ('StanceLabel', String),
        ('ConfidenceLevel', Float64)])

In [22]:
llm_full_stance_df.shape[0]

16044

In [23]:
# Check the number of confidence levels above a certain threshold (e.g., 0.8) in the LLM full stance dataset
print(llm_full_stance_df.filter(pl.col("ConfidenceLevel") >= 0.8).shape)
print(llm_full_stance_df.filter(pl.col("ConfidenceLevel") <= 0.5).shape)
print(llm_full_stance_df.filter((pl.col("ConfidenceLevel") >= 0.5) & (pl.col("ConfidenceLevel") < 0.8)).shape)

(10681, 10)
(14, 10)
(402, 10)


In [20]:
# Get number of unique values in the stance labbel column
llm_full_stance_df.select(pl.col("StanceLabel").n_unique()).to_dicts

<bound method DataFrame.to_dicts of shape: (1, 1)
┌─────────────┐
│ StanceLabel │
│ ---         │
│ u32         │
╞═════════════╡
│ 3           │
└─────────────┘>

In [21]:
# Get the count of each stance label using the value_counts() method
stance_counts = llm_full_stance_df["StanceLabel"].value_counts()
stance_counts

StanceLabel,count
str,u32
"""Favor""",3071
"""Against""",7832
"""Neither""",5141


In [22]:
llm_full_stance_df.head()

UserId,TargetEntity,Context,LLMAnswerContent,LLMAnswerUsage,SourcePosts,Spans,Reason,StanceLabel,ConfidenceLevel
i64,str,list[str],str,str,list[str],list[str],str,str,f64
9,"""Harris""","[""Assuming Harris is the pick. Biden might have endorsed her but I sure as shit hope there's been arrangements to make her the nominee with no fuss."", ""I mean so do I, there's several people I'd want over Harris. That's...not the same as refusing to vote for or support Harris in THIS election. Someone criticizing Harris for specific positions does not automatically equal ""will not vote for her"""", … ""Jill Stein is trash and cannot be trusted. https://anonymous""]","""{ ""source_tweets"": [ …","""{'input_tokens': 888, 'output_…","[""Assuming Harris is the pick. Biden might have endorsed her but I sure as shit hope there's been arrangements to make her the nominee with no fuss."", ""I mean so do I, there's several people I'd want over Harris. That's...not the same as refusing to vote for or support Harris in THIS election."", … ""Cannot express how badly this whole 'Dump Biden' thing could go if it also pulls in Harris""]","[""hope there's been arrangements to make her the nominee with no fuss"", ""not the same as refusing to vote for or support Harris in THIS election"", … ""Cannot express how badly this whole 'Dump Biden' thing could go if it also pulls in Harris""]","""The user expresses a preferenc…","""Favor""",0.85
9,"""Trump""","[""Gonna fucking Battle Royale us into a Trump presidency"", ""I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement."", … ""Hell I'm still seeing BRUCE SPRINGSTEEN takes that are ""too bad he went woke and liberal!""""]","""{ ""source_tweets"": [ …","""{'input_tokens': 810, 'output_…","[""Gonna fucking Battle Royale us into a Trump presidency"", ""I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement."", … ""Any undecided voter who's paid even the slightest attention is a Trump voter trying to justify their vote.""]","[""Gonna fucking Battle Royale us into a Trump presidency"", ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"", … ""Any undecided voter who's paid even the slightest attention is a Trump voter trying to justify their vote""]","""The tweets express a critical …","""Against""",0.95
10,"""Harris""","[""Kamala/Harris"", ""Kamala Harris is very good at this."", … ""Republicans attacks against Kamala thus far: - She's hot - She enjoys laughing - She's quirky""]","""{ ""source_tweets"": [ …","""{'input_tokens': 803, 'output_…","[""Kamala Harris is very good at this."", ""I think Kamala is going to win."", … ""They need to run against Elon Musk, Peter Thiel, David Sacks and the rest of these white nationalist billionaires who oppose unions, democracy, and anything which gives working people a say.""]","[""Kamala Harris is very good at this."", ""I think Kamala is going to win."", … ""run against Elon Musk, Peter Thiel, David Sacks and the rest of these white nationalist billionaires""]","""The tweets express strong supp…","""Favor""",0.95
10,"""Trump""","[""Not to get too political on here, but I really hope Donald Trump does not become president again."", ""Nothing but respect for my president 🥥🌴🥥🌴"", … ""AP still has Trump up""]","""{ ""source_tweets"": [""Not t…","""{'input_tokens': 808, 'output_…","[""Not to get too political on here, but I really hope Donald Trump does not become president again."", ""Not to get too political on here, but I really hope that Donald Trump does not win the presidential election."", ""If Donald Trump wins the presidential election after going full-fascist, I think that will be quite bad

In [23]:
llm_val_stance_df.schema


Schema([('UserId', Int64),
        ('TargetEntity', String),
        ('Context', List(String)),
        ('LLMAnswerContent', String),
        ('LLMAnswerUsage', String),
        ('SourcePosts', List(String)),
        ('Spans', List(String)),
        ('Reason', String),
        ('StanceLabel', String),
        ('ConfidenceLevel', Float64)])

In [24]:
human_val_stance_df.schema

Schema([('UserId', Int64), ('Trump', String), ('Harris', String)])

In [25]:
human_val_relevancy_post_df.schema

Schema([('PostId', Int64), ('Trump', Int64), ('Harris', Int64)])

In [26]:
human_val_relevancy_query_df.schema

Schema([('Query', String),
        ('PostId', Int64),
        ('StanceRelevanceLabel', String)])

## User Network Checking

In [78]:
user_network_df.shape[0]

1676895

In [62]:
# Get the count for all interaction types in the user_network_df

interaction_counts = user_network_df.group_by("InteractionType").agg(pl.count()).sort("count", descending=True)
interaction_counts


C:\Users\andre\AppData\Local\Temp\ipykernel_32124\4114211605.py:3: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  interaction_counts = user_network_df.group_by("InteractionType").agg(pl.count()).sort("count", descending=True)


InteractionType,count
str,u32
"""Like""",991317
"""Repost""",525080
"""Quote""",160498


In [73]:
# Get the count of unique users and the number of interaction types they have participated in
unique_user_interaction_counts = user_network_df.group_by("SourceUserId").agg(
    pl.col("InteractionType").n_unique().alias("InteractionTypeCount")
)
unique_user_interaction_counts

SourceUserId,InteractionTypeCount
i64,u32
252867,1
42879,2
254180,3
14431,3
188522,2
…,…
220605,3
276944,2
304600,2


In [79]:
per_user = user_network_df.group_by("SourceUserId").agg(pl.len().alias("n_interactions"))
per_user.describe()

statistic,SourceUserId,n_interactions
str,f64,f64
"""count""",8255.0,8255.0
"""null_count""",0.0,0.0
"""mean""",199922.27232,203.136887
"""std""",116896.815487,277.230345
"""min""",9.0,1.0
"""25%""",97688.0,34.0
"""50%""",198496.0,105.0
"""75%""",302788.0,252.0
"""max""",401391.0,3158.0


## Graph Creation Testing

The code under this section of the notebook is untested and unverified. Double check before being tested.

Simple idea of how we create the data needed for the GNN: User Network Graph creation (likes and reposts) --> Connect this to the LLM full stance using the user id. <br>
    - Verify if they have the same id types (int64) and if they can be connect to each other

In [24]:
import polars as pl

core = user_post_history_df["UserId"].unique()

reposts = (
    user_post_history_df
    .filter(pl.col("IsRepost"))
    .filter(pl.col("SourceUserId").is_in(core)) # keep core users only
    .filter(pl.col("UserId") != pl.col("SourceUserId"))  # drop self-loops
    .group_by("UserId", "SourceUserId")
    .agg(pl.len().alias("weight"))
    .rename({"UserId": "src", "SourceUserId": "dst"})
)

C:\Users\andre\AppData\Local\Temp\ipykernel_80864\2874847685.py:8: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("SourceUserId").is_in(core)) # keep core users only


In [25]:
reposts

src,dst,weight
i64,i64,u32
372290,89682,1
399299,125916,2
188907,398103,1
255402,115408,1
73585,84421,2
…,…,…
107716,321436,2
115408,308729,1
348360,199739,2


In [26]:
net = pl.read_parquet(str(data_path / "user_network_dataset.parquet"))  # use the actual filename/format from Zenodo
likes = (
    net.filter(pl.col("InteractionType") == "Like")  # adjust column names to match the file
    .select(src="SourceUserId", dst="TargetUserId", weight="Count")
)

In [27]:
likes

src,dst,weight
i64,i64,i64
9,9,1
9,13873,1
9,14511,1
9,62997,1
9,92773,3
…,…,…
401391,385828,4
401391,392266,1
401391,394655,1


Also do the same for quotes to test all possible interactions (also replies but would have to manually derive the connections in `user_post_history`)

In [28]:
quotes = (
    net.filter(pl.col("InteractionType") == "Quote")
    .select(src="SourceUserId", dst="TargetUserId", weight="Count")
)

In [29]:
quotes

src,dst,weight
i64,i64,i64
9,9,42
9,10,2
9,2549,1
9,3125,1
9,4109,1
…,…,…
401391,384107,3
401391,384773,1
401391,398501,2


In [30]:
replies = (
    user_post_history_df
    .filter(pl.col("IsReply"))
    .filter(pl.col("ParentUserId").is_in(core))
    .filter(pl.col("UserId") != pl.col("ParentUserId"))
    .group_by("UserId", "ParentUserId")
    .agg(pl.len().alias("weight"))
    .rename({"UserId": "src", "ParentUserId": "dst"})
)

C:\Users\andre\AppData\Local\Temp\ipykernel_80864\411258742.py:4: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("ParentUserId").is_in(core))


In [31]:
import torch

nodes = core.sort()
idx = {u: i for i, u in enumerate(nodes.to_list())}

def to_edge_index(edges):
    s = [idx[u] for u in edges["src"]]
    d = [idx[u] for u in edges["dst"]]
    return torch.tensor([s, d]), torch.tensor(edges["weight"].to_list(), dtype=torch.float)

edge_index, edge_weight = to_edge_index(likes)  # or reposts

#### Build graph using NetworkX

In [32]:
# NOTE: For now, the graph construction is separate for each interaction type
# We can convert this to a single multi-relational graph later on if needed

In [33]:
import networkx as nx

def build_graph(df: pl.DataFrame) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_weighted_edges_from(
        df.select("src", "dst", "weight").iter_rows(),
        weight="weight"
    )
    return G

In [34]:
reposts_graph = build_graph(reposts)
likes_graph = build_graph(likes)
quotes_graph = build_graph(quotes)
replies_graph = build_graph(replies)

**Attach stance labels**

In [35]:
llm = pl.read_parquet(str(data_path / "llm_annotated_full_user_stance_dataset.parquet"))

In [36]:
print(llm["TargetEntity"].value_counts())
print(llm["StanceLabel"].value_counts())
print(llm["ConfidenceLevel"].value_counts())

shape: (2, 2)
┌──────────────┬───────┐
│ TargetEntity ┆ count │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ Trump        ┆ 8022  │
│ Harris       ┆ 8022  │
└──────────────┴───────┘
shape: (3, 2)
┌─────────────┬───────┐
│ StanceLabel ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ Neither     ┆ 5141  │
│ Against     ┆ 7832  │
│ Favor       ┆ 3071  │
└─────────────┴───────┘
shape: (13, 2)
┌─────────────────┬───────┐
│ ConfidenceLevel ┆ count │
│ ---             ┆ ---   │
│ f64             ┆ u32   │
╞═════════════════╪═══════╡
│ 0.98            ┆ 323   │
│ 0.85            ┆ 886   │
│ 0.75            ┆ 25    │
│ 0.8             ┆ 145   │
│ 0.7             ┆ 300   │
│ …               ┆ …     │
│ 1.0             ┆ 1     │
│ 0.9             ┆ 2003  │
│ 0.5             ┆ 14    │
│ 0.6             ┆ 61    │
│ 0.65            ┆ 2     │
└─────────────────┴───────┘


In [37]:
llm_stances = (
    llm.select("UserId", "TargetEntity", "StanceLabel")
    .pivot(
        on="TargetEntity",
        index="UserId",
        values=["StanceLabel"],
        aggregate_function="first"
    )
)
llm_stances.head()

UserId,Harris,Trump
i64,str,str
9,"""Favor""","""Against"""
10,"""Favor""","""Against"""
49,"""Against""","""Against"""
105,"""Neither""","""Against"""
144,"""Against""","""Against"""


In [38]:
# Get the aggregate count of TargetEntities and StanceLabels
aggregated_count_labels = (
    llm.group_by("TargetEntity", "StanceLabel")
    .agg(pl.count("StanceLabel").alias("count"))
    .sort("count", descending=True)
)
aggregated_count_labels


TargetEntity,StanceLabel,count
str,str,u32
"""Trump""","""Against""",6759
"""Harris""","""Neither""",4223
"""Harris""","""Favor""",2726
"""Harris""","""Against""",1073
"""Trump""","""Neither""",918
"""Trump""","""Favor""",345


In [39]:
attrs = {
    r["UserId"]: {"Trump": r["Trump"], "Harris": r["Harris"]}
    for r in llm_stances.iter_rows(named=True)
}

In [40]:
nx.set_node_attributes(reposts_graph, attrs)
nx.set_node_attributes(likes_graph, attrs)
nx.set_node_attributes(quotes_graph, attrs)
nx.set_node_attributes(replies_graph, attrs)

In [41]:
interaction_graphs = [reposts_graph, likes_graph, quotes_graph, replies_graph]
graph_names = ["Reposts", "Likes", "Quotes", "Replies"]
stance_users = set(llm["UserId"].unique()) 

for G in interaction_graphs:
    graph_users = set(G.nodes())

    labeled_nodes = sum(1 for n in G if G.nodes[n].get("Trump") is not None or G.nodes[n].get("Harris") is not None)
    unlabeled_nodes = G.number_of_nodes() - labeled_nodes

    print(f"Graph: {graph_names[interaction_graphs.index(G)]}")
    print("--------------------------------------------------------------")
    print(f"{labeled_nodes}/{G.number_of_nodes()} nodes have a Trump or Harris label") # made this more general since the count before is just the same with both entities
    print(f"{unlabeled_nodes}/{G.number_of_nodes()} nodes are unlabeled") # users in the graph data are not in the stance dataset
    print(f"{len(stance_users - graph_users)} users in the stance dataset are not in the graph data") # users with stances that never interacted with that type of interaction
    print("--------------------------------------------------------------")

Graph: Reposts
--------------------------------------------------------------
7678/8164 nodes have a Trump or Harris label
486/8164 nodes are unlabeled
344 users in the stance dataset are not in the graph data
--------------------------------------------------------------
Graph: Likes
--------------------------------------------------------------
7937/8454 nodes have a Trump or Harris label
517/8454 nodes are unlabeled
85 users in the stance dataset are not in the graph data
--------------------------------------------------------------
Graph: Quotes
--------------------------------------------------------------
6873/7315 nodes have a Trump or Harris label
442/7315 nodes are unlabeled
1149 users in the stance dataset are not in the graph data
--------------------------------------------------------------
Graph: Replies
--------------------------------------------------------------
7768/8241 nodes have a Trump or Harris label
473/8241 nodes are unlabeled
254 users in the stance dataset 

Since there are users that interact with users involved in the POLITISKY24 dataset, we can try to implement a semi-supervised ml model using a GNN to determine the stances of such users based on their structure in the graph. This might also prove the theories for majority illusion and homophily.

In [42]:
llm.schema

Schema([('UserId', Int64),
        ('TargetEntity', String),
        ('Context', List(String)),
        ('LLMAnswerContent', String),
        ('LLMAnswerUsage', String),
        ('SourcePosts', List(String)),
        ('Spans', List(String)),
        ('Reason', String),
        ('StanceLabel', String),
        ('ConfidenceLevel', Float64)])

In [43]:
# Check for duplicate stance labels for the same user and target entity
duplicate_count = llm.group_by(["UserId", "TargetEntity"]).len().filter(pl.col("len") > 1).height
print(f"Duplicate stance labels for the same user and target entity: {duplicate_count}")

# Check for missing values in the TargetEntity and StanceLabel columns
missing_count = llm.filter(
    pl.col("TargetEntity").is_null() 
    | pl.col("StanceLabel").is_null()
    | (pl.col("TargetEntity") == "")
    | (pl.col("StanceLabel") == "")
).height

print(f"Rows missing TargetEntity or StanceLabel: {missing_count}")

Duplicate stance labels for the same user and target entity: 0
Rows missing TargetEntity or StanceLabel: 0


#### Visualization

NetworkX visualization library can get messy with a large amount of nodes and edges, best use the following recommended libraries, or others not mentioned here:
- gephi (export with `nx.write_gexf(graph, "graph.gexf)`)
- ForceAtlas2 (separate clusters better)
- cosmograph (GPU-accelerated interactable view)

In [44]:
import os
import networkx as nx


def category(attrs):
    t, h = attrs.get("Trump"), attrs.get("Harris")

    if t is None:
        return "Unlabeled"
    if t == "Favor":
        return "Pro-Trump"
    if h == "Favor":
        return "Pro-Harris"
    if t == "Against":
        return "Anti-Trump"
    if h == "Against":
        return "Anti-Harris"

    return "Neither"


def export_gephi(G, path, nodes=None, max_nodes=None):
    # Select an explicit subset of nodes
    if nodes is not None:
        selected_nodes = set(nodes)
        subgraph = G.subgraph(selected_nodes).copy()
    else:
        subgraph = G.copy()

    # Optionally keep only the highest-degree nodes
    if max_nodes is not None and subgraph.number_of_nodes() > max_nodes:
        highest_degree = sorted(
            subgraph.degree,
            key=lambda item: item[1],
            reverse=True
        )[:max_nodes]

        selected_nodes = [node for node, degree in highest_degree]
        subgraph = subgraph.subgraph(selected_nodes).copy()

    H = nx.DiGraph() if subgraph.is_directed() else nx.Graph()

    for node, attrs in subgraph.nodes(data=True):
        H.add_node(
            str(node),
            Trump=str(attrs.get("Trump") or "None"),
            Harris=str(attrs.get("Harris") or "None"),
            category=category(attrs)
        )

    H.add_edges_from(
        (str(source), str(target))
        for source, target in subgraph.edges()
    )

    output_directory = os.path.dirname(path)
    if output_directory:
        os.makedirs(output_directory, exist_ok=True)

    nx.write_gexf(H, path)

In [45]:
graphs = {
    "reposts": reposts_graph,
    "likes": likes_graph,
    "quotes": quotes_graph,
    "replies": replies_graph,
}

for name, graph in graphs.items():
    export_gephi(
        graph,
        f"graph_visual/{name}_top100.gexf", # try 100 for this since 500 runs slow
        max_nodes=100 
    )

## Link Prediction Testing

In [47]:
!py -m pip install torch-geometric

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------------------------------- -------- 1.0/1.3 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 3.3 MB/s  0:00:00



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [50]:
import importlib
from gnn_link_prediction import run_link_prediction
import gnn_link_prediction  # Replace with your module name

# Modify the source file externally, then run:
importlib.reload(gnn_link_prediction)


<module 'gnn_link_prediction' from 'c:\\Users\\andre\\OneDrive\\Documents\\GitHub\\Thesis_Repository\\politisky\\gnn_link_prediction.py'>

In [51]:
# ---------- 5. Run on all four graphs (in the notebook) ----------
import polars as pl
from gnn_link_prediction import run_link_prediction

results = [
    run_link_prediction(g, n_nodes=1000, name=n)
    for g, n in zip([reposts_graph, quotes_graph],
                    ["Reposts", "Quotes"])
]

[Reposts] subgraph: 1000 nodes, 156680 edges
[Reposts] training set: 94008 edges
[Reposts] validation set: 31336 edges, test set: 31336 edges
  epoch  10 | loss 0.5631 | val AUC 0.8375 AP 0.8227
  epoch  20 | loss 0.5077 | val AUC 0.8623 AP 0.8528
  epoch  30 | loss 0.4869 | val AUC 0.8749 AP 0.8664
  epoch  40 | loss 0.4698 | val AUC 0.8850 AP 0.8735
  epoch  50 | loss 0.4608 | val AUC 0.8901 AP 0.8796
  epoch  60 | loss 0.4543 | val AUC 0.8919 AP 0.8806
  epoch  70 | loss 0.4502 | val AUC 0.8993 AP 0.8891
  epoch  80 | loss 0.4417 | val AUC 0.9041 AP 0.8953
  epoch  90 | loss 0.4274 | val AUC 0.9034 AP 0.8940
  epoch 100 | loss 0.4271 | val AUC 0.9095 AP 0.9011
  epoch 110 | loss 0.4166 | val AUC 0.9112 AP 0.9028
  epoch 120 | loss 0.4158 | val AUC 0.9123 AP 0.9042
  epoch 130 | loss 0.4160 | val AUC 0.9128 AP 0.9036
  epoch 140 | loss 0.4108 | val AUC 0.9120 AP 0.9035
  epoch 150 | loss 0.4047 | val AUC 0.9133 AP 0.9048
  epoch 160 | loss 0.4046 | val AUC 0.9143 AP 0.9055
  epoch 17

## Major Topics Discussed in the Dataset

In [53]:
!py -m pip install BERTopic

   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   --------------------- ------------------ 1.0/1.9 MB 6.8 MB/s eta 0:00:01
   -------------------------- ------------- 1.3/1.9 MB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 1.9/1.9 MB 3.3 MB/s  0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.7 MB 9.7 MB/s eta 0:00:01
   -------- ------------------------------- 2.1/9.7 MB 6.1 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.7 MB 5.7 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.7 MB 5.5 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.7 MB 5.6 MB/s eta 0:00:01
   --------------------------- ------------ 6.6/9.7 MB 5.6 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.7 MB 5.6 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.7 MB 5.7 MB/s eta 0:00:01
   --------------------------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
# Topic modeling using every posts in user history dataset

from bertopic import BERTopic

posts = user_post_history_df.select("Content").to_series().to_list()
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(posts)

# View the extracted topics
topic_info = topic_model.get_topic_info()
print(topic_info)

# Get top words for a specific topic (e.g., Topic 0)
print(topic_model.get_topic(0))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3724.45it/s]


KeyboardInterrupt: 